# Doctor Assistant — Classifier Evaluation Lab (Google Colab)

This notebook evaluates **one chest classifier at a time** on pinned, patient-identified NIH ChestX-ray14 data. It is for iterative model development—not the mixed system demo.

Workflow:

1. Run both models on the **validation** split.
2. Review per-label AUROC/AUPRC, calibration, subgroup results, and ranked errors.
3. Improve/retrain the custom model and rerun validation under a new `RUN_NAME`.
4. Select per-label thresholds from validation only.
5. Set `RUN_FINAL_TEST=True` once, after the model and thresholds are frozen.

> Do not tune against the final test output. A successful notebook run establishes a reproducible research benchmark, not clinical validity.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Evaluation configuration

Choose a GPU runtime in **Runtime → Change runtime type**. Set `REPO_REF` to the branch or tag containing the evaluation scripts. Keep a distinct `RUN_NAME` for every model revision.

In [ ]:
from pathlib import Path

# Edit REPO_REF when needed; leave REPO_URL construction unchanged so a Markdown
# hyperlink cannot accidentally be passed to `git clone`.
REPO_SLUG = 'Hamza09Hamza/doctor_assistant'
REPO_URL = f'https://github.com/{REPO_SLUG}.git'
REPO_REF = 'classifier-evaluation-colab'
REPO_DIR = Path('/content/doctor_assistant')

DRIVE_ROOT = Path('/content/drive/MyDrive/doctor_assistant')
CUSTOM_CHECKPOINT = DRIVE_ROOT / 'checkpoints/chest_xray/best.pt'
RUN_NAME = "custom_chest_validation_2000"

RUN_XRV_BASELINE = True
RUN_CUSTOM_MODEL = True
RUN_THRESHOLD_SELECTION = True
RUN_FINAL_TEST = False             # turn on only after model + thresholds are frozen

VALID_IMAGES = 2000
TEST_IMAGES = 2000
SEED = 42
BOOTSTRAP_SAMPLES = 500
SUBGROUP_MIN_IMAGES = 50

# Must match the architecture used to create CUSTOM_CHECKPOINT.
CUSTOM_BACKBONE = 'timm:densenet121'
CUSTOM_IMAGE_SIZE = 320

# Research operating target. Change deliberately and record the rationale.
SENSITIVITY_TARGET = 0.85
SPECIFICITY_FLOOR = 0.60
MIN_VALIDATION_POSITIVES = 20
MIN_VALIDATION_NEGATIVES = 20

CACHE_DIR = Path('/content/doctor_assistant_eval_cache')
ARTIFACT_DIR = DRIVE_ROOT / 'evaluations' / RUN_NAME

print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Custom checkpoint: {CUSTOM_CHECKPOINT}')

## 2 · Mount Drive, pull the exact code revision, and install dependencies

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        'This notebook needs a Google Colab kernel. In VS Code, connect the notebook '
        'to a Colab runtime and rerun this cell.'
    ) from exc

drive.mount('/content/drive')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import os, shutil, subprocess, sys

def run_git(*arguments, cwd=None):
    command = ['git', *map(str, arguments)]
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode:
        detail = result.stderr.strip() or result.stdout.strip() or 'no error text returned'
        raise RuntimeError(f"Git command failed:\n  {' '.join(command)}\n\n{detail}")
    return result.stdout.strip()

if any(character in REPO_URL for character in '[]()'):
    raise ValueError(
        f'REPO_URL must be a plain Git URL, not a Markdown link: {REPO_URL!r}'
    )

# Fail early with a useful message when the branch has not been pushed yet.
remote_ref = run_git('ls-remote', '--heads', REPO_URL, REPO_REF)
if not remote_ref:
    raise RuntimeError(
        f'Remote branch {REPO_REF!r} was not found at {REPO_URL}. '\
        f'Push it from VS Code with: git push -u origin {REPO_REF}'
    )

# A failed clone may leave /content/doctor_assistant without a .git directory.
# It is an ephemeral Colab working copy, so remove only that exact incomplete path.
if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    print(f'Removing incomplete Colab clone: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_git('clone', '--branch', REPO_REF, '--single-branch', REPO_URL, REPO_DIR)
else:
    if run_git('-C', REPO_DIR, 'status', '--porcelain'):
        raise RuntimeError(
            f'{REPO_DIR} has local Colab changes. Restart the runtime or preserve '
            'those changes before refreshing the repository.'
        )
    run_git('-C', REPO_DIR, 'fetch', 'origin', REPO_REF)
    local_branches = run_git('-C', REPO_DIR, 'branch', '--list', REPO_REF)
    if local_branches:
        run_git('-C', REPO_DIR, 'checkout', REPO_REF)
    else:
        run_git('-C', REPO_DIR, 'checkout', '-b', REPO_REF, '--track', f'origin/{REPO_REF}')
    run_git('-C', REPO_DIR, 'merge', '--ff-only', f'origin/{REPO_REF}')

REPO_COMMIT = run_git('-C', REPO_DIR, 'rev-parse', 'HEAD')
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

required_script = REPO_DIR / 'scripts/benchmark_chest_classifier.py'
if not required_script.is_file():
    raise RuntimeError(
        f'{required_script} is missing. Set REPO_REF to the branch containing the '
        'classifier-evaluation work and rerun this cell.'
    )
print(f'Repository commit: {REPO_COMMIT}')

In [3]:
%%capture
!python -m pip install -q \
    'monai>=1.3' 'nibabel>=5.2' 'timm>=0.9' \
    'scipy>=1.12' 'scikit-learn>=1.4' 'scikit-image>=0.22' \
    'pydantic>=2.6' 'Pillow>=10' 'pandas>=2' 'matplotlib>=3.8' \
    'datasets>=2.18' 'huggingface_hub>=0.23' torchxrayvision
print('Evaluation dependencies installed.')

## 3 · Preflight the GPU and custom checkpoint

The notebook refuses to evaluate random custom weights. If the checkpoint is absent, disable `RUN_CUSTOM_MODEL` or upload the real `best.pt` to the configured Drive path.

In [4]:
import json, platform, torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}; PyTorch: {torch.__version__}; Python: {platform.python_version()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

if RUN_CUSTOM_MODEL:
    if not CUSTOM_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f'Real checkpoint not found: {CUSTOM_CHECKPOINT}. '
            'Upload best.pt or set RUN_CUSTOM_MODEL=False.'
        )
    checkpoint = torch.load(CUSTOM_CHECKPOINT, map_location='cpu', weights_only=False)
    if not checkpoint.get('class_names'):
        raise ValueError('Checkpoint has no class_names; label order cannot be verified safely.')
    print('Custom checkpoint preflight:')
    print(f"  epoch={checkpoint.get('epoch')}  best_metric={checkpoint.get('best_metric')}")
    print(f"  classes={checkpoint['class_names']}")

manifest = {
    'run_name': RUN_NAME,
    'repo_url': REPO_URL,
    'repo_ref': REPO_REF,
    'repo_commit': REPO_COMMIT,
    'custom_checkpoint': str(CUSTOM_CHECKPOINT) if RUN_CUSTOM_MODEL else None,
    'valid_images': VALID_IMAGES,
    'test_images': TEST_IMAGES,
    'seed': SEED,
}
(ARTIFACT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')

Device: cuda; PyTorch: 2.11.0+cu128; Python: 3.12.13
GPU: Tesla T4
Custom checkpoint preflight:
  epoch=18  best_metric=0.7386955546072498
  classes=['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


383

## 4 · Helpers and artifact paths

In [5]:
import subprocess

XRV_VALID = ARTIFACT_DIR / 'xrv_validation.json'
CUSTOM_VALID = ARTIFACT_DIR / 'custom_validation.json'
THRESHOLDS = ARTIFACT_DIR / 'selected_thresholds.json'
FINAL_TEST = ARTIFACT_DIR / 'final_test.json'

def run_checked(arguments):
    print('$', ' '.join(str(value) for value in arguments))
    subprocess.run([str(value) for value in arguments], check=True)

def benchmark_command(model, split, count, output, threshold_artifact=None):
    command = [
        sys.executable, REPO_DIR / 'scripts/benchmark_chest_classifier.py',
        '--model', model, '--split', split, '--n', count,
        '--seed', SEED, '--device', DEVICE,
        '--cache-dir', CACHE_DIR,
        '--bootstrap-samples', BOOTSTRAP_SAMPLES,
        '--subgroup-min-images', SUBGROUP_MIN_IMAGES,
        '--output', output,
    ]
    if model == 'custom':
        command += [
            '--checkpoint', CUSTOM_CHECKPOINT,
            '--backbone', CUSTOM_BACKBONE,
            '--image-size', CUSTOM_IMAGE_SIZE,
        ]
    if threshold_artifact is not None:
        command += ['--threshold-artifact', threshold_artifact]
    return command

## 5 · Validation baseline — pretrained TorchXRayVision

This establishes the public pretrained baseline on the exact cohort used for the custom model. It does not train or modify the model.

In [6]:
if RUN_XRV_BASELINE:
    run_checked(benchmark_command('xrv', 'valid', VALID_IMAGES, XRV_VALID))
else:
    print('XRV validation skipped.')

$ /usr/bin/python3 /content/doctor_assistant/scripts/benchmark_chest_classifier.py --model xrv --split valid --n 2000 --seed 42 --device cuda --cache-dir /content/doctor_assistant_eval_cache --bootstrap-samples 500 --subgroup-min-images 50 --output /content/drive/MyDrive/doctor_assistant/evaluations/custom_chest_validation_2000/xrv_validation.json


## 6 · Validation candidate — your custom checkpoint

The custom checkpoint is evaluated independently with the same seed, sample count, dataset revision, metrics, and error-analysis rules.

In [7]:
if RUN_CUSTOM_MODEL:
    run_checked(benchmark_command('custom', 'valid', VALID_IMAGES, CUSTOM_VALID))
else:
    print('Custom validation skipped.')

$ /usr/bin/python3 /content/doctor_assistant/scripts/benchmark_chest_classifier.py --model custom --split valid --n 2000 --seed 42 --device cuda --cache-dir /content/doctor_assistant_eval_cache --bootstrap-samples 500 --subgroup-min-images 50 --output /content/drive/MyDrive/doctor_assistant/evaluations/custom_chest_validation_2000/custom_validation.json --checkpoint /content/drive/MyDrive/doctor_assistant/checkpoints/chest_xray/best.pt --backbone timm:densenet121 --image-size 320


## 7 · Compare scorecards and find the weaknesses

For rare findings, AUPRC is often more informative than AUROC. The operating metrics shown here still use the unvalidated global `0.5` threshold; use them to diagnose score behavior, not as a deployment claim.

In [8]:
import pandas as pd
from IPython.display import display

def load_report(path):
    return json.loads(Path(path).read_text())

reports = {}
if XRV_VALID.is_file(): reports['xrv'] = load_report(XRV_VALID)
if CUSTOM_VALID.is_file(): reports['custom'] = load_report(CUSTOM_VALID)
if not reports:
    raise RuntimeError('No validation reports found; run sections 5 or 6 first.')

macro_rows = []
label_tables = {}
for model_name, report in reports.items():
    scorecard = report['scorecard']
    row = {'model': model_name, **scorecard['macro'], **scorecard['study_level']}
    ci = scorecard.get('confidence_intervals', {}).get('macro', {})
    row['auroc_95ci'] = (
        f"{ci['auroc']['low']:.3f}–{ci['auroc']['high']:.3f}"
        if ci.get('auroc', {}).get('low') is not None else 'n/a'
    )
    macro_rows.append(row)
    table = pd.DataFrame(scorecard['per_label']).T.reset_index(names='label')
    table.insert(0, 'model', model_name)
    label_tables[model_name] = table

print('Study-level and macro scorecard')
display(pd.DataFrame(macro_rows).set_index('model'))

for model_name, table in label_tables.items():
    print(f'Per-label weaknesses — {model_name}')
    display(table.sort_values(['auprc', 'auroc'], na_position='first'))

if {'xrv', 'custom'} <= reports.keys():
    columns = ['label', 'positives', 'prevalence', 'auroc', 'auprc', 'brier', 'ece']
    comparison = label_tables['xrv'][columns].merge(
        label_tables['custom'][columns], on=['label', 'positives', 'prevalence'],
        suffixes=('_xrv', '_custom'),
    )
    comparison['auroc_delta_custom_minus_xrv'] = comparison.auroc_custom - comparison.auroc_xrv
    comparison['auprc_delta_custom_minus_xrv'] = comparison.auprc_custom - comparison.auprc_xrv
    print('Paired cohort comparison (descriptive deltas; inspect confidence intervals)')
    display(comparison.sort_values('auprc_delta_custom_minus_xrv'))

Study-level and macro scorecard


,auroc,auprc,brier,ece,sensitivity,specificity,ppv,npv,f1,dataset_normal_images,dataset_abnormal_images,normal_any_false_positive,normal_mean_findings,abnormal_any_true_label_detected,exact_match_accuracy,auroc_95ci
model,,,,,,,,,,,,,,,,
xrv,0.757798,0.183912,0.364105,0.526633,0.972765,0.201226,0.065305,0.987618,0.115647,1040,960,1.000000,10.488462,0.973958,0.00,0.727–0.781
custom,0.732251,0.132641,0.045392,0.007490,0.000558,0.999959,0.666667,0.947094,0.001103,1040,960,0.000962,0.000962,0.002083,0.52,0.703–0.755


Per-label weaknesses — xrv


,model,label,images,positives,negatives,prevalence,auroc,auprc,brier,ece,threshold,sensitivity,specificity,ppv,npv,f1,tp,fp,tn,fn
12,xrv,Pneumonia,2000.0,25.0,1975.0,0.0125,0.765995,0.037797,0.486226,0.662424,0.5,1.000000,0.118987,0.014164,1.000000,0.027933,25.0,1740.0,235.0,0.0
5,xrv,Emphysema,2000.0,45.0,1955.0,0.0225,0.697874,0.050637,0.472160,0.665974,0.5,1.000000,0.006138,0.022636,1.000000,0.044270,45.0,1943.0,12.0,0.0
6,xrv,Fibrosis,2000.0,27.0,1973.0,0.0135,0.693173,0.053324,0.407010,0.610872,0.5,1.000000,0.063355,0.014400,1.000000,0.028391,27.0,1848.0,125.0,0.0
3,xrv,Edema,2000.0,39.0,1961.0,0.0195,0.800194,0.053907,0.285908,0.439384,0.5,1.000000,0.381948,0.031175,1.000000,0.060465,39.0,1212.0,749.0,0.0
2,xrv,Consolidation,2000.0,81.0,1919.0,0.0405,0.751510,0.098115,0.384819,0.569480,0.5,0.975309,0.153205,0.046362,0.993243,0.088515,79.0,1625.0,294.0,2.0
11,xrv,Pleural_Thickening,2000.0,59.0,1941.0,0.0295,0.738340,0.107368,0.413431,0.609533,0.5,0.983051,0.057187,0.030720,0.991071,0.059579,58.0,1830.0,111.0,1.0
13,xrv,Pneumothorax,2000.0,96.0,1904.0,0.0480,0.713492,0.107903,0.400993,0.590506,0.5,1.000000,0.026786,0.049256,1.000000,0.093888,96.0,1853.0,51.0,0.0
10,xrv,Nodule,2000.0,109.0,1891.0,0.0545,0.706582,0.215389,0.477273,0.652102,0.5,1.000000,0.001058,0.054555,1.000000,0.103465,109.0,1889.0,2.0,0.0
0,xrv,Atelectasis,2000.0,227.0,1773.0,0.1135,0.750372,0.236432,0.368898,0.507236,0.5,0.982379,0.165257,0.130945,0.986532,0.231088,223.0,1480.0,293.0,4.0
8,xrv,Infiltration,2000.0,354.0,1646.0,0.1770,0.650555,0.266118,0.380803,0.485610,0.5,0.957627,0.068651,0.181090,0.882812,0.304582,339.0,1533.0,113.0,15.0


Per-label weaknesses — custom


,model,label,images,positives,negatives,prevalence,auroc,auprc,brier,ece,threshold,sensitivity,specificity,ppv,npv,f1,tp,fp,tn,fn
12,custom,Pneumonia,2000.0,25.0,1975.0,0.0125,0.692638,0.026481,0.012291,0.001103,0.5,0.000000,1.000000,NaN,0.987500,0.000000,0.0,0.0,1975.0,25.0
6,custom,Fibrosis,2000.0,27.0,1973.0,0.0135,0.690676,0.029432,0.013270,0.000912,0.5,0.000000,1.000000,NaN,0.986500,0.000000,0.0,0.0,1973.0,27.0
7,custom,Hernia,2000.0,5.0,1995.0,0.0025,0.793183,0.046063,0.002477,0.000824,0.5,0.000000,1.000000,NaN,0.997500,0.000000,0.0,0.0,1995.0,5.0
3,custom,Edema,2000.0,39.0,1961.0,0.0195,0.827927,0.074384,0.018716,0.008125,0.5,0.000000,1.000000,NaN,0.980500,0.000000,0.0,0.0,1961.0,39.0
2,custom,Consolidation,2000.0,81.0,1919.0,0.0405,0.726780,0.081285,0.038224,0.006657,0.5,0.000000,1.000000,NaN,0.959500,0.000000,0.0,0.0,1919.0,81.0
11,custom,Pleural_Thickening,2000.0,59.0,1941.0,0.0295,0.749238,0.082877,0.027987,0.003442,0.5,0.000000,1.000000,NaN,0.970500,0.000000,0.0,0.0,1941.0,59.0
9,custom,Mass,2000.0,107.0,1893.0,0.0535,0.649446,0.096667,0.050030,0.007562,0.5,0.000000,1.000000,NaN,0.946500,0.000000,0.0,0.0,1893.0,107.0
1,custom,Cardiomegaly,2000.0,53.0,1947.0,0.0265,0.747914,0.102664,0.025179,0.006121,0.5,0.000000,1.000000,NaN,0.973500,0.000000,0.0,0.0,1947.0,53.0
5,custom,Emphysema,2000.0,45.0,1955.0,0.0225,0.774834,0.108567,0.021294,0.003881,0.5,0.000000,1.000000,NaN,0.977500,0.000000,0.0,0.0,1955.0,45.0
10,custom,Nodule,2000.0,109.0,1891.0,0.0545,0.677963,0.174429,0.049427,0.006953,0.5,0.000000,1.000000,NaN,0.945500,0.000000,0.0,0.0,1891.0,109.0


Paired cohort comparison (descriptive deltas; inspect confidence intervals)


,label,positives,prevalence,auroc_xrv,auprc_xrv,brier_xrv,ece_xrv,auroc_custom,auprc_custom,brier_custom,ece_custom,auroc_delta_custom_minus_xrv,auprc_delta_custom_minus_xrv
7,Hernia,5.0,0.0025,0.798997,0.316372,0.113827,0.241427,0.793183,0.046063,0.002477,0.000824,-0.005815,-0.270309
9,Mass,107.0,0.0535,0.798308,0.294571,0.413804,0.594688,0.649446,0.096667,0.050030,0.007562,-0.148861,-0.197905
1,Cardiomegaly,53.0,0.0265,0.893760,0.282226,0.263400,0.424341,0.747914,0.102664,0.025179,0.006121,-0.145846,-0.179563
4,Effusion,256.0,0.1280,0.850024,0.454610,0.228921,0.319288,0.778582,0.311304,0.099702,0.019325,-0.071441,-0.143306
10,Nodule,109.0,0.0545,0.706582,0.215389,0.477273,0.652102,0.677963,0.174429,0.049427,0.006953,-0.028619,-0.040959
11,Pleural_Thickening,59.0,0.0295,0.738340,0.107368,0.413431,0.609533,0.749238,0.082877,0.027987,0.003442,0.010898,-0.024490
6,Fibrosis,27.0,0.0135,0.693173,0.053324,0.407010,0.610872,0.690676,0.029432,0.013270,0.000912,-0.002497,-0.023892
2,Consolidation,81.0,0.0405,0.751510,0.098115,0.384819,0.569480,0.726780,0.081285,0.038224,0.006657,-0.024730,-0.016831
12,Pneumonia,25.0,0.0125,0.765995,0.037797,0.486226,0.662424,0.692638,0.026481,0.012291,0.001103,-0.073357,-0.011315
0,Atelectasis,227.0,0.1135,0.750372,0.236432,0.368898,0.507236,0.709582,0.241541,0.094813,0.008926,-0.040791,0.005109


## 8 · Subgroups and ranked error cases

Inspect view-position, gender, and age-group performance. Small groups are marked insufficient rather than assigned unstable metrics.

In [9]:
ERROR_MODEL = 'custom' if 'custom' in reports else 'xrv'
ERROR_LABEL = 'Pneumothorax'   # change to a weak/important label from the table above

report = reports[ERROR_MODEL]
subgroup_rows = []
for field, groups in report['subgroups'].items():
    for group, values in groups.items():
        if 'status' in values:
            subgroup_rows.append({'field': field, 'group': group, **values})
        else:
            subgroup_rows.append({
                'field': field, 'group': group,
                'images': values['images'], 'patients': values['patients'],
                'macro_auroc': values['macro']['auroc'],
                'macro_auprc': values['macro']['auprc'],
                'normal_any_fp': values['study_level']['normal_any_false_positive'],
            })
display(pd.DataFrame(subgroup_rows))

print(f'Ranked {ERROR_MODEL} errors for {ERROR_LABEL}')
display(pd.DataFrame(report['ranked_errors'][ERROR_LABEL]['false_negatives']).assign(kind='FN'))
display(pd.DataFrame(report['ranked_errors'][ERROR_LABEL]['false_positives']).assign(kind='FP'))

,field,group,images,patients,macro_auroc,macro_auprc,normal_any_fp
0,gender,F,874,793,0.738374,0.145500,0.002137
1,gender,M,1126,977,0.720980,0.128909,0.000000
2,view_position,AP,832,714,0.708072,0.122853,0.002564
3,view_position,PA,1168,1114,0.732398,0.138285,0.000000
4,age_group,18_to_39,541,486,0.738199,0.158880,0.000000
5,age_group,40_to_64,1064,942,0.729785,0.140498,0.000000
6,age_group,65_plus,307,267,0.705080,0.127581,0.006711
7,age_group,under_18,88,80,0.758653,0.310777,0.000000


Ranked custom errors for Pneumothorax


,sample_id,score,kind
0,valid:1937:patient:12626,0.005518,FN
1,valid:1962:patient:1006,0.010362,FN
2,valid:614:patient:14129,0.011542,FN
3,valid:1790:patient:10936,0.011964,FN
4,valid:538:patient:16429,0.013029,FN
5,valid:1196:patient:11356,0.014996,FN
6,valid:1471:patient:10767,0.015645,FN
7,valid:1530:patient:9866,0.015981,FN
8,valid:827:patient:5986,0.016846,FN
9,valid:638:patient:16109,0.018402,FN


,kind


## 9 · Select thresholds from validation only

Choose the candidate to freeze. Threshold export is refused when a label lacks enough validation examples or cannot meet the declared sensitivity/specificity constraints.

In [10]:
SELECTED_MODEL = 'custom' if RUN_CUSTOM_MODEL else 'xrv'
selected_validation = CUSTOM_VALID if SELECTED_MODEL == 'custom' else XRV_VALID

if RUN_THRESHOLD_SELECTION:
    run_checked([
        sys.executable, REPO_DIR / 'scripts/select_chest_thresholds.py',
        '--benchmark', selected_validation,
        '--sensitivity-target', SENSITIVITY_TARGET,
        '--specificity-floor', SPECIFICITY_FLOOR,
        '--min-positives', MIN_VALIDATION_POSITIVES,
        '--min-negatives', MIN_VALIDATION_NEGATIVES,
        '--output', THRESHOLDS,
    ])
    threshold_report = load_report(THRESHOLDS)
    threshold_table = pd.DataFrame(threshold_report['selection']['per_label'])
    display(threshold_table)
    print('Exportable:', threshold_report['thresholds_complete'])
    print('Blocked labels:', threshold_report['blocked_labels'])
else:
    print('Threshold selection skipped.')

$ /usr/bin/python3 /content/doctor_assistant/scripts/select_chest_thresholds.py --benchmark /content/drive/MyDrive/doctor_assistant/evaluations/custom_chest_validation_2000/custom_validation.json --sensitivity-target 0.85 --specificity-floor 0.6 --min-positives 20 --min-negatives 20 --output /content/drive/MyDrive/doctor_assistant/evaluations/custom_chest_validation_2000/selected_thresholds.json


,label,threshold,sensitivity,specificity,positives,negatives,supported,meets_constraints
0,Atelectasis,0.103060,0.713656,0.596165,227,1773,True,False
1,Cardiomegaly,0.019383,0.735849,0.612738,53,1947,True,False
2,Consolidation,0.033314,0.765432,0.605003,81,1919,True,False
3,Edema,0.017814,0.871795,0.696073,39,1961,True,True
4,Effusion,0.097270,0.832031,0.603784,256,1744,True,False
5,Emphysema,0.013873,0.822222,0.574936,45,1955,True,False
6,Fibrosis,0.011077,0.814815,0.501267,27,1973,True,False
7,Hernia,0.500000,0.000000,1.000000,5,1995,False,False
8,Infiltration,0.164766,0.641243,0.597813,354,1646,True,False
9,Mass,0.035455,0.850467,0.398838,107,1893,True,False


Exportable: False
Blocked labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


## 10 · Frozen held-out test — explicit opt-in

This cell will not run unless `RUN_FINAL_TEST=True` and every per-label threshold passed the validation support/constraint gate. After running it, do not change the model or thresholds and present the same test result as final.

In [11]:
if not RUN_FINAL_TEST:
    print('Final held-out test is OFF. Continue improving on validation.')
else:
    if not THRESHOLDS.is_file():
        raise FileNotFoundError('Run threshold selection first.')
    threshold_report = load_report(THRESHOLDS)
    if not threshold_report['thresholds_complete']:
        raise RuntimeError(
            'Threshold artifact is incomplete. Increase validation support or revise '
            'the declared constraints before touching the test split.'
        )
    run_checked(benchmark_command(
        SELECTED_MODEL, 'test', TEST_IMAGES, FINAL_TEST,
        threshold_artifact=THRESHOLDS,
    ))
    final_report = load_report(FINAL_TEST)
    display(pd.DataFrame([{
        'model': SELECTED_MODEL,
        **final_report['scorecard']['macro'],
        **final_report['scorecard']['study_level'],
    }]).set_index('model'))
    print(f'Frozen final artifact: {FINAL_TEST}')

Final held-out test is OFF. Continue improving on validation.


## What to do after each validation round

- Record the weakest labels by AUPRC, false negatives, and calibration—not only macro AUROC.
- Review whether the weakness is data scarcity, label noise, preprocessing, thresholding, or representation capacity.
- Make one controlled training/model change, save a new checkpoint, change `RUN_NAME`, and rerun the same validation cohort.
- Do not promote a model merely because one aggregate metric improved; check per-label regressions and subgroup gaps.
- Move to the next classifier only after the current classifier has a frozen checkpoint, reproducible configuration, complete threshold artifact, and untouched held-out result.